# Análisis crítico de la segmentación

En este capítulo respondemos las seis preguntas críticas del bloque de segmentación con la evidencia de los capítulos {doc}`13_seg_resultados` y {doc}`14_seg_ablaciones_robustez`. Cada respuesta cita la cifra que la sostiene y su intervalo; donde la evidencia no alcanza, lo decimos.

In [1]:
import pandas as pd
from IPython.display import Markdown, display

from project.book_utils import ci, exists, need, pct, style

style()
pd.set_option("display.precision", 3)
S = need("analysis/seg_stats/stats.pkl")
L = S["labels"]
SUM = S["summary"]
CT = {"astro": "astrocitos", "cort": "neuronas corticales", "shsy5y": "SH-SY5Y"}
md = lambda t: display(Markdown(t))  # noqa: E731

## 1. ¿Qué modelo funciona mejor, para qué tipo celular y régimen, y por qué?

In [2]:
best = SUM.ap.idxmax()
md(f"Por AP@[.5:.95] medio en test, el mejor modelo es **{L[best]}** con {ci(SUM.ap[best], SUM.ap_lo[best], SUM.ap_hi[best])}.")
st_ = S["stratified"]
rows = []
for col in ("cell_type", "densidad", "tamaño"):
    t = st_[col]
    for lvl, g in t.groupby("nivel"):
        w = g.sort_values("ap", ascending=False).iloc[0]
        rows.append(dict(estrato=col.replace("cell_type", "tipo celular"), nivel=CT.get(lvl, lvl), mejor=L[w.model],
                         AP=ci(w.ap, w.lo, w.hi), segundo=L[g.sort_values("ap", ascending=False).iloc[1].model] if len(g) > 1 else "—"))
display(pd.DataFrame(rows).style.hide(axis="index"))
h3 = st_["h3"]
md(f"El orden de los modelos {'es el mismo' if h3['identical'] else 'cambia'} entre tipos celulares (W de Kendall = {h3['kendall_w']:.2f}; "
   f"W = 1 es acuerdo total).")

Por AP@[.5:.95] medio en test, el mejor modelo es **Cellpose ajustado** con 0.253 [0.226, 0.281].

estrato,nivel,mejor,AP,segundo
tipo celular,astrocitos,Cellpose cero-shot,"0.159 [0.126, 0.192]",Cellpose ajustado
tipo celular,neuronas corticales,Cellpose ajustado,"0.331 [0.292, 0.368]",Mask R-CNN
tipo celular,SH-SY5Y,Cellpose ajustado,"0.217 [0.198, 0.238]",Mask R-CNN
densidad,alta,Cellpose ajustado,"0.189 [0.163, 0.213]",U-Net++
densidad,baja,Cellpose ajustado,"0.298 [0.245, 0.348]",Mask R-CNN
densidad,media,Cellpose ajustado,"0.274 [0.223, 0.325]",Mask R-CNN
tamaño,grandes,Cellpose ajustado,"0.153 [0.122, 0.185]",Cellpose cero-shot
tamaño,medianas,Cellpose ajustado,"0.322 [0.286, 0.358]",Mask R-CNN
tamaño,pequeñas,Cellpose ajustado,"0.286 [0.238, 0.335]",Mask R-CNN


El orden de los modelos cambia entre tipos celulares (W de Kendall = 0.82; W = 1 es acuerdo total).

In [3]:
ct_t = st_["cell_type"]
def winner(lvl):
    g = ct_t[ct_t.nivel == lvl].sort_values("ap", ascending=False)
    return g.iloc[0].model if len(g) else None
boundary = {"unetpp_main", "hover_main", "cellpose_zeroshot", "cellpose_finetune"}
boxes = {"maskrcnn_main", "sam_box"}
w_sh = winner("shsy5y")
if w_sh in boundary:
    md(f"En SH-SY5Y gana {L[w_sh]}, un método que separa instancias con una señal explícita de frontera o de flujo. Es consistente con el "
       "mecanismo que proponemos: estas neuronas forman agregados densos de células pequeñas con neuritas finas que se tocan, y las cajas "
       "solapadas de un detector compiten en la supresión de no máximos, mientras que una frontera predicha separa células en contacto.")
elif w_sh in boxes:
    md(f"En SH-SY5Y gana {L[w_sh]}, un método basado en cajas, a pesar de los agregados densos. La supresión de no máximos no fue el cuello "
       "de botella que esperábamos; la taxonomía de la sección 3 muestra qué tipo de error domina en los demás modelos.")
w_as = winner("astro")
if w_as is not None:
    a_ap = ct_t[(ct_t.nivel == "astro") & (ct_t.model == w_as)].ap.iloc[0]
    md(f"En astrocitos gana {L[w_as]} con AP {a_ap:.3f}. Los astrocitos son grandes, de forma irregular y con anotaciones poligonales gruesas "
       "(capítulo {doc}`10_seg_eda`); el techo de AP lo fija en parte la anotación, porque un contorno preciso del modelo no "
       "coincide con un polígono grueso a umbrales de IoU altos.")

En SH-SY5Y gana Cellpose ajustado, un método que separa instancias con una señal explícita de frontera o de flujo. Es consistente con el mecanismo que proponemos: estas neuronas forman agregados densos de células pequeñas con neuritas finas que se tocan, y las cajas solapadas de un detector compiten en la supresión de no máximos, mientras que una frontera predicha separa células en contacto.

En astrocitos gana Cellpose cero-shot con AP 0.159. Los astrocitos son grandes, de forma irregular y con anotaciones poligonales gruesas (capítulo {doc}`10_seg_eda`); el techo de AP lo fija en parte la anotación, porque un contorno preciso del modelo no coincide con un polígono grueso a umbrales de IoU altos.

## 2. ¿Coinciden los *rankings* en píxeles e instancias?

In [4]:
h2 = S["h2"]
if h2.get("spearman_auc"):
    r, p = h2["spearman_auc"]
    rb, pb = h2["spearman_bacc"]
    md(f"Entre los {h2['n_models']} modelos con probabilidad por píxel, la correlación de Spearman entre el orden por AUC-ROC de píxel y el "
       f"orden por AP es ρ = {r:.2f} (p = {p:.2g}); con *balanced accuracy*, ρ = {rb:.2f} (p = {pb:.2g}). "
       f"**H2 {'se refuta' if h2['refuted'] else 'no se refuta'}** (el umbral pre-registrado de refutación es ρ > 0.9).")
    sat = [k for k, v in h2["saturated"].items() if v]
    if sat:
        md("Las métricas " + " y ".join(sat).replace("auc_roc", "AUC-ROC").replace("balanced_accuracy", "*balanced accuracy*")
           + " superan 0.95 en todos los modelos: saturan y no discriminan entre ellos.")
    c = h2["contradictions"]
    if len(c):
        md(f"Por imagen, Dice y AP ordenan al revés a un par de modelos en el {pct(c.frac_contradice.mean())} de los casos en promedio.")

Entre los 7 modelos con probabilidad por píxel, la correlación de Spearman entre el orden por AUC-ROC de píxel y el orden por AP es ρ = 0.75 (p = 0.052); con *balanced accuracy*, ρ = 0.54 (p = 0.22). **H2 no se refuta** (el umbral pre-registrado de refutación es ρ > 0.9).

Por imagen, Dice y AP ordenan al revés a un par de modelos en el 25.9 % de los casos en promedio.

Para el uso práctico (contar células, medir su área y seguirlas en el tiempo), la métrica relevante es la de instancias: un error de fusión entre dos células cuenta como un acierto casi completo en Dice de píxel y como dos fallos en AP. Recomendamos AP@[.5:.95] como métrica principal, PQ como resumen que separa calidad de detección (RQ) y de forma (SQ), y las métricas de píxel solo como diagnóstico del mapa de primer plano.

## 3. ¿Qué errores dominan y cómo se relacionan con los datos?

In [5]:
tx = S["taxonomy"]
tx = tx[tx.model == best].set_index("cell_type")
fr = tx[["frac_omitida", "frac_fusionada", "frac_dividida", "frac_borde", "frac_falsa"]].rename(
    columns=lambda c: c.replace("frac_", "")).rename(index=CT)
display(fr)
dom = fr.idxmax(axis=1)
md("Error dominante de " + L[best] + " por tipo celular (fracción de instancias reales): "
   + "; ".join(f"{t}: {d} ({fr.loc[t, d]:.2f})" for t, d in dom.items()) + ".")

,omitida,fusionada,dividida,borde,falsa
cell_type,,,,,
astrocitos,0.526,0.014,0.008,0.144,0.027
neuronas corticales,0.201,0.012,0.004,0.043,0.043
SH-SY5Y,0.168,0.111,0.023,0.083,0.033


Error dominante de Cellpose ajustado por tipo celular (fracción de instancias reales): astrocitos: omitida (0.53); neuronas corticales: omitida (0.20); SH-SY5Y: omitida (0.17).

In [6]:
from scipy.stats import spearmanr  # noqa: E402

TM = need("seg/eval/test_metrics.pkl")
g = TM[TM.model == best].groupby("id")[["n_gt", "err_omitida", "err_fusionada", "err_dividida"]].mean()
rows_ = []
for e, nombre in (("err_omitida", "omisiones"), ("err_fusionada", "fusiones"), ("err_dividida", "divisiones")):
    rho, pv = spearmanr(g.n_gt, g[e] / g.n_gt.clip(lower=1))
    rows_.append((nombre, rho, pv))
txt = "; ".join(f"{n}: ρ = {r:.2f} (p = {p:.2g})" for n, r, p in rows_)
sube = [n for n, r, p in rows_ if r > 0 and p < 0.05]
md(f"Correlación de Spearman entre la densidad (células anotadas por imagen) y la fracción de cada error en {L[best]}, con "
   f"{len(g)} imágenes de test: {txt}. "
   + (f"Con más contactos entre células crecen las {' y las '.join(sube)}: es el patrón esperable cuando la señal que separa "
      "instancias (la frontera predicha, el campo de flujo o la caja del detector) no distingue dos células que se tocan, o marca "
      "dos centros en una célula alargada." if sube else
      "Ninguna fracción de error crece de forma significativa con la densidad en este test."))
top_div = fr["dividida"].idxmax()
md(f"La mayor fracción de divisiones está en {top_div} ({fr.loc[top_div, 'dividida']:.2f})"
   + (", la línea con neuritas largas: una célula alargada con dos ensanchamientos puede recibir dos centros y partirse." if top_div == CT["shsy5y"] else "."))

Correlación de Spearman entre la densidad (células anotadas por imagen) y la fracción de cada error en Cellpose ajustado, con 93 imágenes de test: omisiones: ρ = 0.15 (p = 0.15); fusiones: ρ = 0.64 (p = 7.2e-12); divisiones: ρ = 0.46 (p = 3e-06). Con más contactos entre células crecen las fusiones y las divisiones: es el patrón esperable cuando la señal que separa instancias (la frontera predicha, el campo de flujo o la caja del detector) no distingue dos células que se tocan, o marca dos centros en una célula alargada.

La mayor fracción de divisiones está en SH-SY5Y (0.02), la línea con neuritas largas: una célula alargada con dos ensanchamientos puede recibir dos centros y partirse.

Las detecciones falsas incluyen tanto ruido como células reales sin anotar: la inspección visual del capítulo {doc}`10_seg_eda` encontró errores evidentes de anotación en una parte de los recortes revisados, así que una fracción de los «falsos positivos» no es error del modelo.

## 4. Limitaciones, sesgos del conjunto de datos y amenazas a la validez

| Tipo | Amenaza | Qué hicimos | Validez de la conclusión |
|---|---|---|---|
| Interna | Fuga entre particiones por imágenes del mismo pocillo o placa | Partición agrupada por `sample_id` y estratificada por tipo celular | Válida si las imágenes de un mismo grupo no comparten placa con otros grupos |
| Interna | Selección del posproceso con test | Rejilla del posproceso congelada en validación; test evaluado una vez | — |
| Interna | Presupuesto de entrenamiento desigual entre modelos | Mismo número máximo de épocas por familia, *early stopping* en validación; desviaciones registradas. Al 25 % de los datos, Mask R-CNN recibió muchos menos pasos de gradiente que la U-Net++ (tabla de H1 en el capítulo {doc}`13_seg_resultados`) | Las diferencias pequeñas pueden deberse al presupuesto; el contraste de H1 al 25 % mezcla preentrenamiento y presupuesto de optimización |
| Interna | Réplicas de las ablaciones recortadas | Variantes con una semilla rotuladas como exploratorias; con dos, solo IC por imagen y rango de la diferencia entre semillas | Las variantes de una semilla no admiten conclusión confirmatoria; las de dos no tienen estimación de la varianza entre semillas |
| Externa | Un microscopio, un laboratorio, tres líneas celulares, contraste de fase | LOCTO y perturbaciones de imagen | No generaliza a otras modalidades sin reevaluar |
| Constructo | Anotaciones con errores y polígonos gruesos en astrocitos | Inspección visual y taxonomía de errores | El AP de astrocitos tiene un techo por debajo de 1 |
| Constructo | AP@[.5:.95] penaliza el desplazamiento de borde de células pequeñas más que el de grandes | Reportamos AP50, PQ y HD95 junto al AP | El orden por AP mezcla detección y precisión de borde |
| Conclusión estadística | Imágenes de test por tipo celular pocas para estratos finos | IC *bootstrap*, Holm, modelo mixto con imagen como grupo | Diferencias por estrato con IC solapados quedan sin decidir |

## 5. ¿Cómo afectó el preprocesamiento y qué decisiones resultaron irrelevantes?

In [7]:
ab = S["ablations"]
for g in ("pérdida", "fracción"):
    if g in ab:
        t0 = ab[g]["table"]
        expl = t0[t0.seeds < 2]
        t = t0[t0.seeds >= 2]
        incons = t.signo_consistente.eq(False)              # el signo cambia entre semillas: no sostiene «mejor» ni «peor»
        eq = (t.lo >= -0.02) & (t.hi <= 0.02)                 # equivalente: IC completo dentro de ±δ
        sig = (t.hi < 0) | (t.lo > 0)                         # IC que excluye 0

        def _tag(r):
            return f"{int(r.seeds)} semillas, " + ("mismo signo en todas" if r.signo_consistente else "signo distinto entre semillas")

        lst = lambda m: ", ".join(f"{r.variante} ({round(r['diff'], 3) + 0.0:+.3f}; {_tag(r)})" for _, r in t[m].iterrows())  # noqa: E731
        parts = []
        if (eq & ~sig).any():
            parts.append(f"equivalentes a la referencia (IC dentro de ±0.02): {lst(eq & ~sig)}")
        if (eq & sig).any():
            parts.append(f"distintas de la referencia pero dentro de ±0.02, sin relevancia práctica: {lst(eq & sig)}")
        if (~eq & (t.hi < 0) & ~incons).any():
            parts.append(f"peores (IC entero bajo 0, fuera de ±0.02 y mismo signo en todas las semillas): {lst(~eq & (t.hi < 0) & ~incons)}")
        if (~eq & (t.lo > 0) & ~incons).any():
            parts.append(f"mejores (IC entero sobre 0 y mismo signo en todas las semillas): {lst(~eq & (t.lo > 0) & ~incons)}")
        if (~eq & sig & incons).any():
            parts.append(f"sin decidir: IC por imagen que excluye 0 pero signo distinto entre semillas: {lst(~eq & sig & incons)}")
        if (~eq & ~sig).any():
            parts.append(f"sin decidir (IC que cruza 0 y sale de ±0.02): {lst(~eq & ~sig)}")
        if len(expl):
            parts.append("exploratorias, una semilla y sin conclusión confirmatoria: " + ", ".join(
                f"{r.variante} ({round(r['diff'], 3) + 0.0:+.3f} [{r.lo:.3f}, {r.hi:.3f}])" for _, r in expl.iterrows()))
        md(f"**{g.capitalize()}** (referencia {ab[g]['reference'].replace(' (ref.)', '')}, AP {ab[g]['ref_ap']:.3f}): " + "; ".join(parts) + ".")
if exists("eda/seg/normalization.pkl"):
    nz = need("eda/seg/normalization.pkl")
    md("La normalización por percentiles de cada imagen reduce la diferencia de contraste entre los tres tipos celulares (capítulo "
       "{doc}`11_seg_preprocesamiento`) y, como efecto lateral, anula los cambios afines de brillo y contraste (perturbaciones del "
       "capítulo {doc}`14_seg_ablaciones_robustez`). No la sometimos a ablación con entrenamiento completo por presupuesto, así que su "
       "efecto sobre el AP en datos limpios queda sin medir.")

**Pérdida** (referencia CE + Dice, AP 0.214): equivalentes a la referencia (IC dentro de ±0.02): focal + Tversky (+0.001; 2 semillas, signo distinto entre semillas); distintas de la referencia pero dentro de ±0.02, sin relevancia práctica: CE (+0.005; 2 semillas, mismo signo en todas); peores (IC entero bajo 0, fuera de ±0.02 y mismo signo en todas las semillas): Dice (-0.036; 2 semillas, mismo signo en todas).

**Fracción** (referencia 100 %, AP 0.214): exploratorias, una semilla y sin conclusión confirmatoria: 50 % (+0.003 [-0.002, 0.008]), 25 % (+0.000 [-0.005, 0.005]), 10 % (-0.004 [-0.009, 0.000]).

La normalización por percentiles de cada imagen reduce la diferencia de contraste entre los tres tipos celulares (capítulo {doc}`11_seg_preprocesamiento`) y, como efecto lateral, anula los cambios afines de brillo y contraste (perturbaciones del capítulo {doc}`14_seg_ablaciones_robustez`). No la sometimos a ablación con entrenamiento completo por presupuesto, así que su efecto sobre el AP en datos limpios queda sin medir.

## 6. ¿Qué haría el modelo con otro microscopio o protocolo?

In [8]:
lo = S["locto"]
if len(lo):
    for _, r in lo.iterrows():
        md(f"- Sin {CT[r.excluded]} en entrenamiento, el AP en {CT[r.excluded]} pasa de {r.ap_main:.3f} a {r.ap_locto:.3f} "
           f"(diferencia {ci(round(r['diff'], 3) + 0.0, r.lo, r.hi)}, n = {int(r.n)}; "
           + ("exploratoria, 1 semilla" if r.seeds < 2 else
              f"{int(r.seeds)} semillas, diferencia por semilla de {r.diff_seed_min:+.3f} a {r.diff_seed_max:+.3f}") + ").")
if exists("analysis/seg_perturb/rows.pkl"):
    pr = pd.DataFrame(need("analysis/seg_perturb/rows.pkl"))
    g = pr.groupby(["model", "perturbation", "severity"]).ap.mean().reset_index()
    base = g[g.perturbation == "ninguna"].set_index("model").ap
    worst = g[(g.perturbation != "ninguna") & (g.severity == 5)].assign(caida=lambda d: d.model.map(base) - d.ap)
    worst = worst.sort_values("caida", ascending=False).head(4)
    md("Con la severidad máxima, las cuatro caídas más grandes de AP son: "
       + "; ".join(f"{L.get(m, m)} con {k} ({c:.3f})" for m, k, c in zip(worst.model, worst.perturbation, worst.caida)) + ".")

- Sin astrocitos en entrenamiento, el AP en astrocitos pasa de 0.077 a 0.035 (diferencia -0.041 [-0.055, -0.027], n = 21; 2 semillas, diferencia por semilla de -0.044 a -0.038).

- Sin neuronas corticales en entrenamiento, el AP en neuronas corticales pasa de 0.298 a 0.175 (diferencia -0.123 [-0.144, -0.104], n = 48; 2 semillas, diferencia por semilla de -0.131 a -0.113).

- Sin SH-SY5Y en entrenamiento, el AP en SH-SY5Y pasa de 0.166 a 0.048 (diferencia -0.118 [-0.138, -0.099], n = 24; 2 semillas, diferencia por semilla de -0.121 a -0.113).

Con la severidad máxima, las cuatro caídas más grandes de AP son: Mask R-CNN con desenfoque (σ px) (0.228); U-Net++ con desenfoque (σ px) (0.191); Mask R-CNN con ruido gaussiano (σ) (0.140); U-Net++ con ruido gaussiano (σ) (0.132).

Un microscopio distinto cambia el contraste, el ruido, el desenfoque y la escala en píxeles por micra. Las perturbaciones del capítulo {doc}`14_seg_ablaciones_robustez` simulan los tres primeros; la escala no la simulamos y es la más peligrosa para los métodos con posproceso de tamaño fijo (el `min_size` del watershed y las anclas de Mask R-CNN). LOCTO mide algo distinto: una línea celular nueva con el mismo microscopio. Antes de desplegar en otro equipo hace falta un conjunto de validación etiquetado de ese equipo, una recalibración del posproceso con él y el monitoreo de la confianza media por imagen (capítulo {doc}`31_monitoreo_deriva`).